<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/torch/hard/custom-autograd/custom-autgrad-function.ipynb)

Check your work: `pip install -e python/` once from the repo root (on Colab: `!pip install -q "git+https://github.com/Fengyee/TorchLeet#subdirectory=python"`), then `from torchleet import check; check("custom-autograd", ...)`


# Problem: Write a Custom Activation Function with Autograd

### Problem Statement
Implement a **custom activation function**, **Learned-SiLU**, using `torch.autograd.Function`. The activation is SiLU (a.k.a. Swish), \( x \cdot \text{sigmoid}(x) \), scaled by a **learnable slope parameter**. Use it inside a small regression model and train it.

### Requirements
1. **Define the Custom Activation Function**:
   - Implement a custom activation function, **Learned-SiLU**, where the output is calculated as:
     $$
     \text{Learned-SiLU}(x) = \text{slope} \cdot x \cdot \text{sigmoid}(x)
     $$
   - The **slope** should be a learnable parameter.

2. **Autograd Implementation**:
   - Subclass `torch.autograd.Function` and write **both** `forward` and `backward` yourself (derive the derivatives by hand).
   - `backward` must return one gradient per input of `forward` — for `x` *and* for `slope` — each with the shape of that input. `slope` is broadcast over `x` in the forward pass, so its gradient has to be summed back down to `slope`'s shape.
   - Stash the tensors `backward` needs with `ctx.save_for_backward(...)`, not as plain attributes on `ctx`.
   - Verify your `backward` with `torch.autograd.gradcheck` (in float64).

3. **Integrate the Activation Function**:
   - Model: the activation applied to the input feature, followed by a linear layer — `prediction = Linear(Learned-SiLU(x))`.
   - Train it on the synthetic data `y = 2x + 3 + noise` and check that the loss falls.

### Constraints
- The **slope** must be an `nn.Parameter` initialized from the constructor argument, and it must be updated during training.

<details>
  <summary>💡 Hint</summary>
  Some details: https://pytorch.org/tutorials/beginner/examples_autograd/two_layer_net_custom_function.html
</details>


<details>
  <summary>💡 Alternate Implementation?</summary>
  Can be done with nn.Module without implementing backward.
</details>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
# Generate synthetic data
torch.manual_seed(42)
X = torch.rand(100, 1) * 10  # 100 data points between 0 and 10
y = 2 * X + 3 + torch.randn(100, 1)  # Linear relationship with noise

In [ ]:
class LearnedSiLUFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, slope):
        # TODO: save the tensors backward needs (ctx.save_for_backward), then return the output
        ...

    @staticmethod
    def backward(ctx, grad_output):
        # TODO: retrieve the saved tensors and return (grad wrt x, grad wrt slope)
        ...


# Define the Linear Regression Model
class LinearRegressionModel(nn.Module):
    def __init__(self, slope=1):
        super().__init__()
        self.linear = nn.Linear(1, 1)
        self.slope = nn.Parameter(torch.ones(1) * slope)

    def forward(self, x):
        # TODO: apply the custom LearnedSiLUFunction to x, then the linear layer
        ...

# Initialize the model, loss function, and optimizer
model = LinearRegressionModel()
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.005)  # x goes up to 10: lr=0.01 is unstable here

# Training loop
epochs = 1000
for epoch in range(epochs):
    # Forward pass
    predictions = model(X)
    loss = criterion(predictions, y)

    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Log progress every 100 epochs
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}")

In [ ]:
# Check the hand-written backward against finite differences (float64 is required
# for gradcheck's numerical derivative to be accurate enough)
x_check = torch.randn(8, 3, dtype=torch.double, requires_grad=True)
slope_check = torch.tensor([1.5], dtype=torch.double, requires_grad=True)
print("gradcheck passed:", torch.autograd.gradcheck(LearnedSiLUFunction.apply, (x_check, slope_check)))

In [ ]:
# Display the learned parameters
[w, b] = model.linear.parameters()
print(f"Learned weight: {w.item():.4f}, Learned bias: {b.item():.4f}, Learned slope: {model.slope.item():.4f}")

# Testing on new data
X_test = torch.tensor([[4.0], [7.0]])
with torch.no_grad():
    predictions = model(X_test)
    print(f"Predictions for {X_test.tolist()}: {predictions.tolist()}")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("custom-autograd", LearnedSiLUFunction=LearnedSiLUFunction,
      LinearRegressionModel=LinearRegressionModel)